### Porting to Google Colab
The following cell enables this notebook to run from Google Colab as well as from your local machine IDE.<br>
You can change `root_directory` and/or `this_notebook_google_path` to point to the directory in your Google account, which contains this notebook, together with the `imgs` sub-directory and the rest of the files.<br>

In [1]:
import sys
import os
try:
    from google.colab import drive as google_drive # type: ignore
except:
    # no Google Colab --> fall back to local machine
    google_drive = None

if google_drive is not None:
    google_drive_directory = os.path.join('/','content','gdrive')
    google_drive.mount(google_drive_directory)
    all_projects_path = os.path.join(google_drive_directory, 'Othercomputers','My Laptop', 'projects')
elif os.name == 'posix': # Ubuntu
    all_projects_path = os.path.join(os.path.expanduser('~'), 'projects')
elif os.name == 'nt': # Windows
    all_projects_path = os.path.join('d:\\', 'projects')
else:
    raise EnvironmentError("Unsupported operating system")
    
project_path = os.path.join(all_projects_path,'RUNI','Thesis')
assert os.path.exists(project_path), f'Project path {project_path} not found!'
# enable import python files from this notebook's path
sys.path.append(project_path)
# enable reading images and data files from this notebook's path
os.chdir(project_path)

datasets_path = os.path.join(project_path, 'datasets')
assert os.path.exists(datasets_path), f'Datasets path {datasets_path} not found!'

output_path = os.path.join(project_path, 'output')
os.makedirs(output_path, exist_ok=True)
assert os.path.exists(output_path), f'Output path {output_path} not found!'
print(f'Current working directory: {os.getcwd()}')
print(f'Datasets path: {datasets_path}')
print(f'Output path: {output_path}')



Current working directory: /home/dror/projects/RUNI/Thesis
Datasets path: /home/dror/projects/RUNI/Thesis/datasets
Output path: /home/dror/projects/RUNI/Thesis/output


In [2]:
import numpy as np

In [3]:
from python.hpc import HybridArray

Detecting CUDA version prior to importing numba...
Checking nvcc --version...
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Fri Aug 22 15:38:44 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.64.03              Driver Version: 575.64.03      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650        

# Testing random p-values in vector

In [4]:
from python.rare_weak_model.rare_weak_model import random_p_values_series
num_p_values = 100000
seed = 3
data_py = HybridArray()
data_py.realloc(shape=(num_p_values,), dtype=np.float64, use_gpu=False)
random_p_values_series(p_values_output=data_py, seed=seed, use_njit=False)
print(f'{data_py.numpy().mean()=}')


data_njit = HybridArray()
data_njit.realloc(shape=(num_p_values,), dtype=np.float64, use_gpu=False)
random_p_values_series(p_values_output=data_njit, seed=seed, use_njit=True)
print(f'{data_njit.numpy().mean()=}')

data_gpu = HybridArray()
data_gpu.realloc(shape=(num_p_values,), dtype=np.float64, use_gpu=True)
random_p_values_series(p_values_output=data_gpu, seed=seed)
print(f'{data_gpu.numpy().mean()=}')

data_py.numpy().mean()=np.float64(0.5000918424567481)


TypingError: Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'random_integers': Cannot determine Numba type of <class 'type'>

File "python/rare_weak_model/numba_cpu.py", line 57:
    def random_p_values_series_cpu_njit(seed: np.uint64, out: np.ndarray) -> None:
        <source elided>
        work_array = np.empty_like(out, dtype=np.uint64)
        random_integers.random_integers_series(seed=seed, out=work_array)
        ^

During: Pass nopython_type_inference

In [ ]:
from python.rare_weak_model.rare_weak_model import random_p_values_matrix
shape = (5,5)
seed = 0
data_py = HybridArray().realloc(shape=shape, dtype=np.float64, use_gpu=False)
num_steps = 1
num_steps = 10
random_p_values_matrix(p_values_output=data_py, offset_row0=seed, offset_col0=0, num_steps=num_steps, use_njit=False)
print(f'{data_py.numpy()=}')


data_njit = HybridArray().realloc(shape=shape, dtype=np.float64, use_gpu=False)
random_p_values_matrix(p_values_output=data_njit, offset_row0=seed, offset_col0=0, num_steps=num_steps, use_njit=True)
print(f'{data_njit.numpy()=}')

data_gpu = HybridArray().realloc(shape=shape, dtype=np.float64, use_gpu=True)
random_p_values_matrix(p_values_output=data_gpu, offset_row0=seed, offset_col0=0, num_steps=num_steps)
print(f'{data_gpu.numpy()=}')

In [ ]:
from python.rare_weak_model.rare_weak_model import random_modified_p_values_matrix
shape = (5,5)
seed = 0
data_py = HybridArray().realloc(shape=shape, dtype=np.float64, use_gpu=False)
num_steps = 1
num_steps = 10
mu = 1
random_modified_p_values_matrix(p_values_output=data_py, mu=mu, offset_row0=seed, offset_col0=0, num_steps=num_steps, use_njit=False)
print(f'{data_py.numpy()=}')


data_njit = HybridArray().realloc(shape=shape, dtype=np.float64, use_gpu=False)
random_modified_p_values_matrix(p_values_output=data_njit, mu=mu, offset_row0=seed, offset_col0=0, num_steps=num_steps, use_njit=True)
print(f'{data_njit.numpy()=}')

data_gpu = HybridArray().realloc(shape=shape, dtype=np.float64, use_gpu=True)
random_modified_p_values_matrix(p_values_output=data_gpu, mu=mu, offset_row0=seed, offset_col0=0, num_steps=num_steps)
print(f'{data_gpu.numpy()=}')

In [ ]:
from python.rare_weak_model.rare_weak_model import rare_weak_model
shape = (5,5)
n1 = 2
seed = 0
data_py = HybridArray().realloc(shape=shape, dtype=np.float64, use_gpu=False)
counts_py = HybridArray()
num_steps = 1
mu = 1
rare_weak_model(sorted_p_values_output=data_py, cumulative_counts_output=counts_py, mu=mu, n1=n1, num_steps=num_steps, use_njit=False, sort_labels=False)
print(f'Original values N={shape[1]} {n1=}\n{data_py.numpy()}')


rare_weak_model(sorted_p_values_output=data_py, cumulative_counts_output=counts_py, mu=mu, n1=n1, use_njit=False)
print('\nNative Python:')
print(f'data=\n{data_py.numpy()}')
print(f'counts=\n{counts_py.numpy()}')

data_njit = HybridArray().realloc(shape=shape, dtype=np.float64, use_gpu=False)
counts_njit = HybridArray()
rare_weak_model(sorted_p_values_output=data_njit, cumulative_counts_output=counts_njit, mu=mu, n1=n1, use_njit=True)
print('\nNumba NJIT')
print(f'data=\n{data_njit.numpy()}')
print(f'counts=\n{counts_njit.numpy()}')

data_gpu = HybridArray().realloc(shape=shape, dtype=np.float64, use_gpu=True)
counts_gpu = HybridArray()
rare_weak_model(sorted_p_values_output=data_gpu, cumulative_counts_output=counts_gpu, mu=mu, n1=n1)
print('\nNumba CUDA')
print(f'data=\n{data_gpu.numpy()}')
print(f'counts=\n{counts_gpu.numpy()}')